# After the presentation: explore the models

This notebook runs **in your web browser**. Work through the cells from the top using **Shift + Enter** (or choose **Run All**). You do not need to install Python.

### Three experiments to try

1. Change `HOURS_CUTOFF` from `7` to `10` and see how many students the human-written rule flags.
2. Change `MAX_DEPTH` from `3` to `2` or `4` and compare decision-tree accuracy and failure recall.
3. Remove `SelfStudyHoursPerWeek` from `FEATURES` and observe which question the algorithm learns first.

All student records are **fictitious**. A model's prediction is not a diagnosis and must not be used on its own to restrict student support.

If the audience guesses are unknown, keep `?` entries; the notebook will skip scoring intuition until they are filled in. To reproduce your session's Wooclap majority choices, replace the eight `?` values with `Support` or `No support` in display order.


# Can Python help students succeed?
### A hands-on, introduction to machine learning

**Scenario:** You are a data scientist working for a university. Your task is to identify students who *may be at risk of not passing* so that staff can consider whether to offer academic support.

**Your journey:** **Human judgement → Python rules → Decision trees → Compare results.**

No experience of Python is required. Select a code cell and press **Shift + Enter** to run it. In cells labelled **✏️ YOUR TURN**, make a small change, then run the cell again.

| Part | Workshop stage |
|---|---|
| 1. | Predict which eight students might need support |
| 2. | Make a rule-based classifier |
| 3. | Follow a hand-drawn decision tree |
| 4. | Learn why the tree chooses its first question |
| 5. | Grow a tree from historical records |
| 6. | Evaluate human, rule-based and learned predictions |
| 7. | Reflect on mistakes, fairness and support |

> This is a synthetic classroom exercise. **Predicting eventual failure is not the same thing as establishing whether someone needs support.**

## 1 — You are the data scientist  

The university has **200 fictional student records**, each with five fields:

| Field | What it means | Example |
|---|---|---|
| `StudentID` | Anonymous record identifier | S024 |
| `AttendancePct` | Percentage of lectures and labs attended | 50 = attended half |
| `PreviousMarksTotal` | **Sum of two previous assessment marks**, each out of 50 | 65 out of 100 |
| `SelfStudyHoursPerWeek` | Hours spent studying independently **per week**, outside scheduled teaching | 6.5 hours |
| `Outcome` | Whether the student **eventually** passed or failed | Pass / Fail |

The first three numerical columns are the **features** we can use to predict. `Outcome` is the **label** (the historical answer). `StudentID` is an identifier, not a predictive feature.

We will use **150 records for learning** and keep **50 unseen records for testing**. First, examine eight randomly selected records from the testing set. Their pass/fail outcomes are deliberately hidden.

In [ ]:
import pandas as pd
from IPython.display import display

# Keep this notebook and the CSV in the same folder.
students = pd.read_csv('student_data_self_study.csv')

# Reproducible random order: 150 for learning, 50 unseen for testing.
shuffled = students.sample(frac=1, random_state=41).reset_index(drop=True)
training = shuffled.iloc[:150].copy()
testing = shuffled.iloc[150:].copy()

# Eight randomly ordered test records; their outcome labels remain hidden.
focus = testing.sample(n=8, random_state=6).reset_index(drop=True).copy()

print(f'Loaded {len(students)} fictional records.')
print(f'Learning set: {len(training)} students; unseen test set: {len(testing)} students.')
print('Eight students: outcomes HIDDEN')
display(focus[['StudentID', 'AttendancePct', 'PreviousMarksTotal', 'SelfStudyHoursPerWeek']]
        .rename(columns={'AttendancePct': 'Attendance (%)',
                         'PreviousMarksTotal': 'Earlier marks (/100)',
                         'SelfStudyHoursPerWeek': 'Self-study (hours/week)'}))

### YOUR TURN 1 — Decide who to flag

Which of the eight students would you flag for **Support**, and which would you mark **No support**? Discuss your reasoning before seeing the answers.

In the cell below, replace each `"?"` with `"Support"` or `"No support"`. **Keep the quotation marks.** Your answers will be compared with the two algorithms at the end.

In [ ]:
# YOUR TURN: change the eight question marks, in the displayed order.
audience_guesses = [
    "?",  # Student 1
    "?",  # Student 2
    "?",  # Student 3
    "?",  # Student 4
    "?",  # Student 5
    "?",  # Student 6
    "?",  # Student 7
    "?"   # Student 8
]

assert len(audience_guesses) == 8, 'Enter eight guesses.'
assert all(item in ('?', 'Support', 'No support') for item in audience_guesses), (
    'Please use only ?, Support, or No support.')
initial_decisions = focus[['StudentID']].copy()
initial_decisions['Your prediction'] = audience_guesses
display(initial_decisions)

## 2 — Use `if` and `else` to make a rule  

You can make a simple rule that checks **three** pieces of information:

> **IF** attendance is below **75%**, **OR** earlier assessment marks are below **55/100**, **OR** self-study is below **7 hours per week**, predict **Support**. Otherwise, predict **No support**.

Here, the **human chose** the features, numerical cut-offs and the `or` logic. Python just applies the same instruction to everyone.

### YOUR TURN 2 — Edit a cut-off

Run the next cell, then change `HOURS_CUTOFF` from 7 to 5. Does the classification change for anyone? Try a different attendance threshold too.

In [ ]:
# YOUR TURN: change one or more values and run again.
ATTENDANCE_CUTOFF = 75
MARKS_CUTOFF = 55
HOURS_CUTOFF = 7

def rule_based_prediction(student):
    if (student['AttendancePct'] < ATTENDANCE_CUTOFF or
        student['PreviousMarksTotal'] < MARKS_CUTOFF or
        student['SelfStudyHoursPerWeek'] < HOURS_CUTOFF):
        return 'Support'
    else:
        return 'No support'

focus['Rule prediction'] = focus.apply(rule_based_prediction, axis=1)
display(focus[['StudentID', 'AttendancePct', 'PreviousMarksTotal',
               'SelfStudyHoursPerWeek', 'Rule prediction']])

## 3 — The same rule as a decision tree  

A **decision tree** is a series of questions with **yes/no branches**. Our rule can be drawn as:

```text
Is attendance below 75%?
├── YES → Support
└── NO  → Are previous marks below 55?
            ├── YES → Support
            └── NO  → Is self-study below 7 hours/week?
                        ├── YES → Support
                        └── NO  → No support
```

**Discussion:** Someone has 82% attendance, 67 marks out of 100 and studies independently for 3 hours per week. What does the tree predict?

This is a **hand-designed tree**, because we picked the questions and cut-offs ourselves. A *machine-learned* tree instead examines historical examples to decide **which feature to ask about first**, what threshold to use, and what questions to ask next.

In [ ]:
# Optional: change these numbers and try another student.
new_student = {
    'AttendancePct': 82,
    'PreviousMarksTotal': 67,
    'SelfStudyHoursPerWeek': 3
}
print('Prediction:', rule_based_prediction(new_student))

## 4 — How does Python choose the *first* question?  

We now let Python learn from the 150 students for whom the final Pass/Fail outcome is known.

To select the first question, the algorithm:

1. Tries each feature: **attendance**, **previous marks**, **self-study hours**.
2. Tries possible cut-off values for each feature.
3. Calculates how well the cut-off separates historical Pass and Fail labels.
4. Chooses the combination that produces the **greatest Gini reduction**.

**Gini impurity** measures how mixed the answers are within a group: **0** means entirely Pass or entirely Fail; **0.5** means a 50/50 mixture. The best question makes the resulting groups **less mixed**.

You do not need to write the algorithm below. Run it and interpret the table.

In [ ]:
# Historical Pass/Fail labels are used ONLY for learning, never as input features.
training['Failed'] = (training['Outcome'] == 'Fail').astype(int)

FEATURES = ['AttendancePct', 'PreviousMarksTotal', 'SelfStudyHoursPerWeek']
FRIENDLY_NAMES = {'AttendancePct': 'Attendance (%)',
                  'PreviousMarksTotal': 'Previous marks (/100)',
                  'SelfStudyHoursPerWeek': 'Self-study (hours/week)'}

def gini(outcomes):
    """0 means a pure group; 0.5 means half Pass, half Fail."""
    if len(outcomes) == 0:
        return 0.0
    fraction_failing = sum(outcomes) / len(outcomes)
    return 2 * fraction_failing * (1 - fraction_failing)

def best_question(data, features, minimum_per_branch=10):
    """Find feature + threshold with largest Gini improvement."""
    before = gini(data['Failed'])
    best = None
    for feature in features:
        values = sorted(data[feature].unique())
        # A split between neighbouring observed values avoids pointless cut-offs.
        for low, high in zip(values[:-1], values[1:]):
            threshold = (low + high) / 2
            left = data[data[feature] < threshold]
            right = data[data[feature] >= threshold]
            if min(len(left), len(right)) < minimum_per_branch:
                continue
            after = (len(left) * gini(left['Failed']) +
                     len(right) * gini(right['Failed'])) / len(data)
            gain = before - after
            if best is None or gain > best['gain'] + 1e-12:
                best = {'feature': feature, 'threshold': float(threshold),
                        'gain': float(gain), 'left_n': len(left), 'right_n': len(right)}
    return best

options = []
for feature in FEATURES:
    choice = best_question(training, [feature])
    options.append({'Possible FIRST feature': FRIENDLY_NAMES[feature],
                    'Best cut-off (<)': round(choice['threshold'], 2),
                    'Gini reduction (higher is better)': round(choice['gain'], 3)})

print('Which feature gives the largest improvement on the TRAINING set?')
display(pd.DataFrame(options).sort_values(
    'Gini reduction (higher is better)', ascending=False).reset_index(drop=True))

### What difference did the extra feature make?

The next cell compares the best first question with **two features** against the best first question with **three features**. It is the same training data, so the only change is whether **self-study hours** is available to the learner.

**Predict first:** Will the algorithm ask about attendance, earlier marks, or self-study hours? Why might that choice differ from the question a person would ask first?

In [ ]:
old_features = ['AttendancePct', 'PreviousMarksTotal']
old_root = best_question(training, old_features)
new_root = best_question(training, FEATURES)
print(f'Without self-study: FIRST ask whether {FRIENDLY_NAMES[old_root["feature"]]} '
      f'< {old_root["threshold"]:.2f}')
print(f'With self-study:    FIRST ask whether {FRIENDLY_NAMES[new_root["feature"]]} '
      f'< {new_root["threshold"]:.2f}')
print('The algorithm chooses the feature producing the greatest reduction in Gini impurity.')

## 5 — Let Python build the rest of the tree  

After selecting the first question, the algorithm repeats the calculation **separately for each branch**, until it reaches its allowed depth. Each endpoint, or **leaf**, predicts the most common training label in that group.

**YOUR TURN 3:** Run the cell and read the generated questions. Then change `MAX_DEPTH` from **3** to **2** or **4**. What changes?

The helper functions below implement a small decision-tree learner **without using a machine-learning package**. You only need to change the single number.

In [ ]:
def grow_tree(data, features, depth=0, max_depth=3, minimum_per_branch=10):
    failures = int(data['Failed'].sum())
    prediction = 'Support' if failures >= len(data)/2 else 'No support'
    leaf = {'is_leaf': True, 'prediction': prediction,
            'count': len(data), 'failures': failures}
    if depth >= max_depth or data['Failed'].nunique() == 1:
        return leaf

    question = best_question(data, features, minimum_per_branch)
    if question is None or question['gain'] < 0.005:
        return leaf

    f, t = question['feature'], question['threshold']
    left = data[data[f] < t]
    right = data[data[f] >= t]
    return {'is_leaf': False, 'feature': f, 'threshold': t,
            'gain': question['gain'],
            'yes': grow_tree(left, features, depth+1, max_depth, minimum_per_branch),
            'no': grow_tree(right, features, depth+1, max_depth, minimum_per_branch)}

def show_tree(branch, indent=''):
    if branch['is_leaf']:
        print(f'{indent}→ {branch["prediction"]} '
              f'(historical Fail: {branch["failures"]}/{branch["count"]})')
    else:
        print(f'{indent}Is {FRIENDLY_NAMES[branch["feature"]]} '
              f'< {branch["threshold"]:.2f}?')
        print(f'{indent}  YES:')
        show_tree(branch['yes'], indent+'    ')
        print(f'{indent}  NO:')
        show_tree(branch['no'], indent+'    ')

def tree_predict(student, branch):
    if branch['is_leaf']:
        return branch['prediction']
    next_branch = (branch['yes'] if student[branch['feature']] < branch['threshold']
                   else branch['no'])
    return tree_predict(student, next_branch)

# YOUR TURN: change this to 2 or 4 if you want.
MAX_DEPTH = 3
learned_tree = grow_tree(training, FEATURES, max_depth=MAX_DEPTH)
print('DECISION TREE LEARNED FROM 150 HISTORICAL RECORDS')
show_tree(learned_tree)

## 6 — Compare predictions BEFORE revealing outcomes  

We now have **three** different ways of deciding which students to flag:

- **Audience intuition** — based on the eight records you saw first.
- **Rule-based algorithm** — based on your three chosen thresholds.
- **Machine-learned decision tree** — based on the examples in the 150-student training set.

All three approaches predict for the **same eight students**. The real Pass/Fail outcomes remain hidden for one more cell.

In [ ]:
# Refresh predictions in case you edited thresholds or tree depth.
focus['Rule prediction'] = focus.apply(rule_based_prediction, axis=1)
focus['Tree prediction'] = focus.apply(lambda row: tree_predict(row, learned_tree), axis=1)

hidden_results = focus[['StudentID','AttendancePct','PreviousMarksTotal',
                        'SelfStudyHoursPerWeek','Rule prediction','Tree prediction']].copy()
hidden_results.insert(4, 'Your prediction', audience_guesses)
print('The answers are still HIDDEN:')
display(hidden_results)

## 7 — Reveal the answers and test your predictions  

Now we reveal the actual outcomes.

A **true positive (TP)** is a student who *failed* and was flagged for support. A **false positive (FP)** is a student flagged for support who *passed*. A **false negative (FN)** is a student who failed but was *not* flagged. A **true negative (TN)** is someone correctly predicted not to need a warning who passed.

- **Accuracy:** What percentage of all predictions were correct?
- **Recall:** Of students who actually failed, what percentage were flagged?

Recall matters here because a **missed student** could mean a missed opportunity to offer support. False alarms also carry a cost, so more warnings are not automatically better.

In [ ]:
comparison = focus[['StudentID','AttendancePct','PreviousMarksTotal',
                    'SelfStudyHoursPerWeek','Outcome']].copy()
comparison['Your judgement'] = audience_guesses
comparison['Rule-based'] = focus['Rule prediction']
comparison['Decision tree'] = focus['Tree prediction']
comparison = comparison[['StudentID','AttendancePct','PreviousMarksTotal',
                         'SelfStudyHoursPerWeek','Your judgement',
                         'Rule-based','Decision tree','Outcome']]
print('REVEAL: the eight students, including their eventual outcomes')
display(comparison)

def score_predictions(approach, predictions, actual_outcomes):
    pairs = list(zip(predictions, actual_outcomes))
    tp = sum(p == 'Support' and a == 'Fail' for p, a in pairs)
    fp = sum(p == 'Support' and a == 'Pass' for p, a in pairs)
    fn = sum(p == 'No support' and a == 'Fail' for p, a in pairs)
    tn = sum(p == 'No support' and a == 'Pass' for p, a in pairs)
    n = len(pairs)
    assert tp + fp + fn + tn == n, 'Unexpected classification value.'
    return {'Method': approach, 'Correct': tp+tn, 'Total': n,
            'Accuracy (%)': round(100*(tp+tn)/n,1),
            'Recall of failures (%)': round(100*tp/(tp+fn),1) if tp+fn else None,
            'TP':tp,'FP':fp,'FN':fn,'TN':tn}

scores_8 = []
if all(g in ('Support','No support') for g in audience_guesses):
    scores_8.append(score_predictions('Audience', audience_guesses, focus['Outcome']))
else:
    print('Audience score omitted: fill in all eight guesses in YOUR TURN 1 and rerun.')
scores_8.append(score_predictions('Rule-based', focus['Rule prediction'], focus['Outcome']))
scores_8.append(score_predictions('Decision tree', focus['Tree prediction'], focus['Outcome']))
print('FAIR COMPARISON: all scores below use the same eight students')
display(pd.DataFrame(scores_8))

### Understand a confusion matrix

A confusion matrix shows correct decisions and the two types of mistake. **Rows are the eventual outcome; columns are the prediction.**

| | Predicted: Support | Predicted: No support |
|---|---:|---:|
| **Actually failed** | TP (correct warning) | FN (missed warning) |
| **Actually passed** | FP (false alarm) | TN (correct reassurance) |

The following code shows the confusion matrix for the learned decision tree using those eight students.

In [ ]:
tree_score_8 = score_predictions('Tree', focus['Tree prediction'], focus['Outcome'])
confusion_8 = pd.DataFrame(
    [[tree_score_8['TP'], tree_score_8['FN']],
     [tree_score_8['FP'], tree_score_8['TN']]],
    index=['Actually Fail', 'Actually Pass'],
    columns=['Predicted Support', 'Predicted No support'])
display(confusion_8)

### A more reliable check: all 50 unseen students

With only **eight** students, just one different answer changes accuracy by **12.5 percentage points**. We can evaluate the algorithms on all **50** reserved test cases. We cannot score the audience on all 50 because they only predicted eight.

We also build a **two-feature tree** without self-study hours, using the same 150 training students, to investigate whether adding a feature actually improves test performance. This is a classroom illustration, **not** a guarantee that extra features always help.

In [ ]:
# Rule and learned tree predict using ONLY features, not the known Outcomes.
rule_predictions_50 = testing.apply(rule_based_prediction, axis=1)
tree_predictions_50 = testing.apply(lambda row: tree_predict(row, learned_tree), axis=1)

# Extra comparison: a tree trained without the new self-study feature.
two_feature_tree = grow_tree(training, old_features, max_depth=MAX_DEPTH)
two_feature_predictions_50 = testing.apply(
    lambda row: tree_predict(row, two_feature_tree), axis=1)

scores_50 = [
    score_predictions('Human-written rule (3 features)', rule_predictions_50, testing['Outcome']),
    score_predictions('Learned tree (2 features)', two_feature_predictions_50, testing['Outcome']),
    score_predictions('Learned tree (3 features)', tree_predictions_50, testing['Outcome'])
]
print('ALL 50 unseen students: identical test set for every method')
display(pd.DataFrame(scores_50))

print('Discussion: Which model has the highest accuracy? Which flags the most failures?')
print('Did adding self-study hours change the first question or test performance?')

## 8 — Your closing discussion  

1. **Is the learned tree always more accurate than a simple rule?** What changes when `MAX_DEPTH` changes?
2. **What does an additional feature do?** Did the first question change when self-study hours were introduced? Did prediction quality improve?
3. **Which mistake matters more?** Missing a student who later fails, or flagging someone who would have passed?

### What you have learned

- You can turn human judgement into a Python rule with `if` and `else`.
- A decision tree asks questions in sequence; **a machine-learned tree chooses its own features and cut-offs from historical labelled data**.
- Testing must use records that were **not used to train the model**.
- Accuracy and recall tell us different things, and **having more features does not automatically produce a better model**.

> **Real-world caution:** These records are completely invented. Eventual failure is only a proxy for support needs. Providing support might change the outcome; attendance and hours of study do not explain everything. A real university would need human oversight, privacy safeguards and a careful approach to fairness and student wellbeing.